In [3]:
# =============================================
# 1. IMPORT ALL REQUIRED LIBRARIES (UNCHANGED)
# =============================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Machine Learning Libraries
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA

# Classification Algorithms
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.cluster import KMeans
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

# Evaluation Metrics
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                            f1_score, confusion_matrix, classification_report,
                            roc_auc_score, roc_curve)

# GUI Libraries
import gradio as gr
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Additional Utilities
import pickle
import io
import base64
from datetime import datetime
import time

# =============================================
# 2. DATA LOADING AND PREPROCESSING CLASSES
# =============================================

class DiseaseDataset:
    """Base class for handling disease datasets"""

    def __init__(self, name, file_path):
        self.name = name
        self.file_path = file_path
        self.data = None
        self.X = None
        self.y = None
        self.X_train = None
        self.X_test = None
        self.y_train = None
        self.y_test = None
        self.scaler = StandardScaler()
        self.feature_names = []
        self.target_name = ""

    def load_data(self):
        """Load dataset from local CSV file"""
        try:
            self.data = pd.read_csv(self.file_path)
            print(f"✅ {self.name} dataset loaded successfully. Shape: {self.data.shape}")
            return True
        except Exception as e:
            print(f"❌ Error loading {self.name} dataset from {self.file_path}: {e}")
            # Create a dummy dataset if file doesn't exist
            self.create_dummy_data()
            return False

    def create_dummy_data(self):
        """Create dummy data for testing if real data isn't available"""
        print(f"⚠️ Creating dummy data for {self.name}...")
        # Create simple dummy dataset
        np.random.seed(42)
        n_samples = 100
        n_features = 10

        # Create feature data
        X = np.random.randn(n_samples, n_features)
        # Create binary target
        y = np.random.randint(0, 2, n_samples)

        # Create feature names
        self.feature_names = [f'feature_{i}' for i in range(n_features)]
        self.target_name = 'target'

        # Create dataframe
        data_dict = {}
        for i in range(n_features):
            data_dict[f'feature_{i}'] = X[:, i]
        data_dict['target'] = y

        self.data = pd.DataFrame(data_dict)
        self.X = self.data[self.feature_names]
        self.y = self.data[self.target_name]

        print(f"✅ Created dummy data for {self.name}. Shape: {self.data.shape}")

    def preprocess(self):
        """Abstract method for preprocessing"""
        pass

    def split_data(self, test_size=0.2):
        """Split data into training and testing sets"""
        if self.X is not None and self.y is not None:
            self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
                self.X, self.y, test_size=test_size, random_state=42, stratify=self.y
            )
            print(f"✅ {self.name} data split: Train={self.X_train.shape}, Test={self.X_test.shape}")

    def scale_features(self):
        """Scale features using StandardScaler"""
        if self.X_train is not None:
            self.X_train = self.scaler.fit_transform(self.X_train)
            self.X_test = self.scaler.transform(self.X_test)
            print(f"✅ {self.name} features scaled")

    def get_sample_inputs(self):
        """Get sample inputs for GUI"""
        # Default implementation returns empty dict
        return {}

class HeartDiseaseDataset(DiseaseDataset):
    """Heart Disease Dataset Handler"""

    def __init__(self):
        # FIXED: Added error handling for missing files
        super().__init__("Heart Disease", "/content/heart.csv")  # Updated path

    def preprocess(self):
        """Preprocess heart disease dataset"""
        try:
            if self.data is None:
                if not self.load_data():
                    return False

            # Create a copy
            df = self.data.copy()

            # Clean column names (remove extra spaces, special characters)
            df.columns = df.columns.str.strip()

            # Based on your screenshot, identify target column
            # Common target column names: 'target', 'num', 'disease', 'class'
            target_candidates = ['target', 'num', 'disease', 'class', 'Target', 'Disease', 'Class']

            for candidate in target_candidates:
                if candidate in df.columns:
                    self.target_name = candidate
                    break

            if self.target_name == "":
                # If no standard target name found, use last column
                self.target_name = df.columns[-1]

            print(f"Using target column: {self.target_name}")

            # Handle missing values
            if df.isnull().sum().sum() > 0:
                df = df.dropna()

            # Convert target to binary if needed
            unique_targets = df[self.target_name].unique()
            if len(unique_targets) > 2:
                # If multi-class, convert to binary (0 = no disease, 1 = disease)
                df[self.target_name] = df[self.target_name].apply(lambda x: 0 if x == 0 else 1)

            # Separate features and target
            self.y = df[self.target_name]
            self.X = df.drop(columns=[self.target_name])

            # Store feature names
            self.feature_names = self.X.columns.tolist()

            print(f"✅ Heart Disease preprocessed. Features: {len(self.feature_names)}")
            return True
        except Exception as e:
            print(f"❌ Error preprocessing Heart Disease: {e}")
            return False

    def get_sample_inputs(self):
        """Return sample inputs for GUI"""
        # Return default values for heart disease features
        inputs = {
            'age': 52, 'sex': 1, 'cp': 0, 'trestbps': 125,
            'chol': 212, 'fbs': 0, 'restecg': 1, 'thalach': 168,
            'exang': 0, 'oldpeak': 1.0, 'slope': 2, 'ca': 2, 'thal': 3
        }

        # Filter to only include features that exist in our dataset
        if self.feature_names:
            filtered_inputs = {}
            for feat in self.feature_names:
                if feat in inputs:
                    filtered_inputs[feat] = inputs[feat]
                else:
                    # Use default value
                    filtered_inputs[feat] = 0
            return filtered_inputs

        return inputs

class DiabetesDataset(DiseaseDataset):
    """Diabetes Dataset Handler"""

    def __init__(self):
        super().__init__("Diabetes", "/content/diabetes.csv")  # Updated path

    def preprocess(self):
        """Preprocess diabetes dataset"""
        try:
            if self.data is None:
                if not self.load_data():
                    return False

            df = self.data.copy()

            # Clean column names
            df.columns = df.columns.str.strip()

            # Identify target column
            target_candidates = ['Outcome', 'target', 'diabetes', 'class', 'result']

            for candidate in target_candidates:
                if candidate in df.columns:
                    self.target_name = candidate
                    break

            if self.target_name == "":
                self.target_name = df.columns[-1]

            print(f"Using target column: {self.target_name}")

            # Handle zeros as missing values for certain columns
            medical_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
            for col in medical_cols:
                if col in df.columns:
                    df[col] = df[col].replace(0, np.nan)

            # Impute missing values
            imputer = SimpleImputer(strategy='median')
            for col in df.columns:
                if col != self.target_name and df[col].isnull().sum() > 0:
                    df[col] = imputer.fit_transform(df[[col]]).ravel()

            # Separate features and target
            self.y = df[self.target_name]
            self.X = df.drop(columns=[self.target_name])

            self.feature_names = self.X.columns.tolist()

            print(f"✅ Diabetes preprocessed. Features: {len(self.feature_names)}")
            return True
        except Exception as e:
            print(f"❌ Error preprocessing Diabetes: {e}")
            return False

    def get_sample_inputs(self):
        """Return sample inputs for GUI"""
        inputs = {
            'Pregnancies': 6, 'Glucose': 148, 'BloodPressure': 72,
            'SkinThickness': 35, 'Insulin': 0, 'BMI': 33.6,
            'DiabetesPedigreeFunction': 0.627, 'Age': 50
        }

        if self.feature_names:
            filtered_inputs = {}
            for feat in self.feature_names:
                if feat in inputs:
                    filtered_inputs[feat] = inputs[feat]
                else:
                    filtered_inputs[feat] = 0
            return filtered_inputs

        return inputs

class StrokeDataset(DiseaseDataset):
    """Stroke Dataset Handler"""

    def __init__(self):
        super().__init__("Stroke", "/content/healthcare-dataset-stroke-data.csv")  # Updated path

    def preprocess(self):
        """Preprocess stroke dataset"""
        try:
            if self.data is None:
                if not self.load_data():
                    return False

            df = self.data.copy()

            # Clean column names
            df.columns = df.columns.str.strip()

            # Identify target column
            target_candidates = ['stroke', 'target', 'result', 'class', 'Stroke']

            for candidate in target_candidates:
                if candidate in df.columns:
                    self.target_name = candidate
                    break

            if self.target_name == "":
                self.target_name = df.columns[-1]

            print(f"Using target column: {self.target_name}")

            # Handle missing values
            for col in df.columns:
                if df[col].isnull().sum() > 0:
                    if df[col].dtype == 'object':
                        df[col].fillna(df[col].mode()[0], inplace=True)
                    else:
                        df[col].fillna(df[col].median(), inplace=True)

            # Encode categorical variables
            categorical_cols = df.select_dtypes(include=['object']).columns

            # Remove target column if it's categorical
            if self.target_name in categorical_cols:
                categorical_cols = categorical_cols.drop(self.target_name)

            le = LabelEncoder()
            for col in categorical_cols:
                df[col] = le.fit_transform(df[col].astype(str))

            # Encode target if categorical
            if df[self.target_name].dtype == 'object':
                df[self.target_name] = le.fit_transform(df[self.target_name])

            # Separate features and target
            self.y = df[self.target_name]
            self.X = df.drop(columns=[self.target_name])

            self.feature_names = self.X.columns.tolist()

            print(f"✅ Stroke preprocessed. Features: {len(self.feature_names)}")
            return True
        except Exception as e:
            print(f"❌ Error preprocessing Stroke: {e}")
            return False

    def get_sample_inputs(self):
        """Return sample inputs for GUI"""
        inputs = {
            'gender': 1, 'age': 67, 'hypertension': 0, 'heart_disease': 1,
            'ever_married': 1, 'work_type': 2, 'Residence_type': 1,
            'avg_glucose_level': 228.69, 'bmi': 36.6, 'smoking_status': 1
        }

        if self.feature_names:
            filtered_inputs = {}
            for feat in self.feature_names:
                if feat in inputs:
                    filtered_inputs[feat] = inputs[feat]
                else:
                    filtered_inputs[feat] = 0
            return filtered_inputs

        return inputs

class LiverDataset(DiseaseDataset):
    """Liver Disease Dataset Handler"""

    def __init__(self):
        super().__init__("Liver Disease", "/content/indian_liver_patient.csv")  # Updated path

    def preprocess(self):
        """Preprocess liver disease dataset"""
        try:
            if self.data is None:
                if not self.load_data():
                    return False

            df = self.data.copy()

            # Clean column names
            df.columns = df.columns.str.strip()

            # Identify target column
            target_candidates = ['Dataset', 'target', 'result', 'class', 'disease', 'Liver']

            for candidate in target_candidates:
                if candidate in df.columns:
                    self.target_name = candidate
                    break

            if self.target_name == "":
                self.target_name = df.columns[-1]

            print(f"Using target column: {self.target_name}")

            # Handle missing values
            for col in df.columns:
                if df[col].isnull().sum() > 0:
                    if df[col].dtype == 'object':
                        df[col].fillna(df[col].mode()[0], inplace=True)
                    else:
                        df[col].fillna(df[col].median(), inplace=True)

            # Encode categorical variables (like gender)
            categorical_cols = df.select_dtypes(include=['object']).columns

            # Remove target column if it's categorical
            if self.target_name in categorical_cols:
                categorical_cols = categorical_cols.drop(self.target_name)

            le = LabelEncoder()
            for col in categorical_cols:
                df[col] = le.fit_transform(df[col].astype(str))

            # Encode target if categorical
            if df[self.target_name].dtype == 'object':
                df[self.target_name] = le.fit_transform(df[self.target_name])

            # Separate features and target
            self.y = df[self.target_name]
            self.X = df.drop(columns=[self.target_name])

            self.feature_names = self.X.columns.tolist()

            print(f"✅ Liver Disease preprocessed. Features: {len(self.feature_names)}")
            return True
        except Exception as e:
            print(f"❌ Error preprocessing Liver Disease: {e}")
            return False

    def get_sample_inputs(self):
        """Return sample inputs for GUI"""
        inputs = {
            'Age': 65, 'gender': 1, 'Total_Bilirubin': 0.7, 'Direct_Bilirubin': 0.1,
            'Alkaline_Phosphotase': 187, 'Alamine_Aminotransferase': 16,
            'Aspartate_Aminotransferase': 18, 'Total_Protiens': 6.8,
            'Albumin': 3.3, 'Albumin_and_Globulin_Ratio': 0.9
        }

        if self.feature_names:
            filtered_inputs = {}
            for feat in self.feature_names:
                # Try to match with inputs
                matched = False
                for input_key in inputs:
                    if input_key.lower() in feat.lower() or feat.lower() in input_key.lower():
                        filtered_inputs[feat] = inputs[input_key]
                        matched = True
                        break
                if not matched:
                    filtered_inputs[feat] = 0
            return filtered_inputs

        return inputs

class KidneyDiseaseDataset(DiseaseDataset):
    """Chronic Kidney Disease Dataset Handler"""

    def __init__(self):
        super().__init__("Chronic Kidney Disease", "/content/kidney_disease.csv")  # Updated path

    def preprocess(self):
        """Preprocess kidney disease dataset"""
        try:
            if self.data is None:
                if not self.load_data():
                    return False

            df = self.data.copy()

            # Clean column names
            df.columns = df.columns.str.strip()

            # Identify target column
            target_candidates = ['classification', 'target', 'result', 'class', 'ckd', 'disease']

            for candidate in target_candidates:
                if candidate in df.columns:
                    self.target_name = candidate
                    break

            if self.target_name == "":
                self.target_name = df.columns[-1]

            print(f"Using target column: {self.target_name}")

            # Handle missing values
            for col in df.columns:
                if df[col].isnull().sum() > 0:
                    if df[col].dtype == 'object':
                        # Replace '?' with NaN then fill
                        df[col] = df[col].replace('?', np.nan)
                        df[col].fillna(df[col].mode()[0], inplace=True)
                    else:
                        df[col].fillna(df[col].median(), inplace=True)

            # Convert categorical to numerical
            categorical_cols = df.select_dtypes(include=['object']).columns

            # Remove target column if it's categorical
            if self.target_name in categorical_cols:
                categorical_cols = categorical_cols.drop(self.target_name)

            le = LabelEncoder()
            for col in categorical_cols:
                df[col] = le.fit_transform(df[col].astype(str))

            # Encode target if categorical
            if df[self.target_name].dtype == 'object':
                df[self.target_name] = le.fit_transform(df[self.target_name])

            # Separate features and target
            self.y = df[self.target_name]
            self.X = df.drop(columns=[self.target_name])

            self.feature_names = self.X.columns.tolist()

            print(f"✅ Kidney Disease preprocessed. Features: {len(self.feature_names)}")
            return True
        except Exception as e:
            print(f"❌ Error preprocessing Kidney Disease: {e}")
            return False

    def get_sample_inputs(self):
        """Return sample inputs for GUI"""
        inputs = {
            'age': 48, 'bp': 80, 'sg': 1.02, 'al': 1, 'su': 0,
            'rbc': 1, 'pc': 1, 'pcc': 0, 'ba': 0, 'bgr': 121,
            'bu': 36, 'sc': 1.2, 'sod': 137, 'pot': 4.6, 'hemo': 15.4,
            'pcv': 44, 'wc': 7800, 'rc': 5.2, 'htn': 1, 'dm': 1,
            'cad': 0, 'appet': 1, 'pe': 0, 'ane': 0
        }

        if self.feature_names:
            filtered_inputs = {}
            for feat in self.feature_names:
                if feat in inputs:
                    filtered_inputs[feat] = inputs[feat]
                else:
                    filtered_inputs[feat] = 0
            return filtered_inputs

        return inputs

# =============================================
# 3. MACHINE LEARNING MODEL CLASS (UNCHANGED)
# =============================================

class DiseasePredictor:
    """Main class for training and evaluating models"""

    def __init__(self):
        self.models = {}
        self.results = {}
        self.pca_models = {}
        self.kmeans_models = {}
        self.scalers = {}

    def train_models(self, X_train, X_test, y_train, y_test, disease_name):
        """Train all models for a specific disease"""

        results = {}

        # 1. Logistic Regression
        try:
            print(f"Training Logistic Regression for {disease_name}...")
            lr = LogisticRegression(max_iter=1000, random_state=42)
            lr.fit(X_train, y_train)
            lr_pred = lr.predict(X_test)
            results['Logistic Regression'] = {
                'model': lr,
                'predictions': lr_pred,
                'accuracy': accuracy_score(y_test, lr_pred),
                'precision': precision_score(y_test, lr_pred, average='weighted', zero_division=0),
                'recall': recall_score(y_test, lr_pred, average='weighted', zero_division=0),
                'f1': f1_score(y_test, lr_pred, average='weighted', zero_division=0),
                'confusion_matrix': confusion_matrix(y_test, lr_pred)
            }
        except Exception as e:
            print(f"❌ Error training Logistic Regression: {e}")

        # 2. K-Nearest Neighbors (KNN)
        try:
            print(f"Training K-Nearest Neighbors for {disease_name}...")
            knn = KNeighborsClassifier(n_neighbors=5)
            knn.fit(X_train, y_train)
            knn_pred = knn.predict(X_test)
            results['K-Nearest Neighbors'] = {
                'model': knn,
                'predictions': knn_pred,
                'accuracy': accuracy_score(y_test, knn_pred),
                'precision': precision_score(y_test, knn_pred, average='weighted', zero_division=0),
                'recall': recall_score(y_test, knn_pred, average='weighted', zero_division=0),
                'f1': f1_score(y_test, knn_pred, average='weighted', zero_division=0),
                'confusion_matrix': confusion_matrix(y_test, knn_pred)
            }
        except Exception as e:
            print(f"❌ Error training K-Nearest Neighbors: {e}")

        # 3. Naive Bayes
        try:
            print(f"Training Naive Bayes for {disease_name}...")
            nb = GaussianNB()
            nb.fit(X_train, y_train)
            nb_pred = nb.predict(X_test)
            results['Naive Bayes'] = {
                'model': nb,
                'predictions': nb_pred,
                'accuracy': accuracy_score(y_test, nb_pred),
                'precision': precision_score(y_test, nb_pred, average='weighted', zero_division=0),
                'recall': recall_score(y_test, nb_pred, average='weighted', zero_division=0),
                'f1': f1_score(y_test, nb_pred, average='weighted', zero_division=0),
                'confusion_matrix': confusion_matrix(y_test, nb_pred)
            }
        except Exception as e:
            print(f"❌ Error training Naive Bayes: {e}")

        # 4. Decision Tree
        try:
            print(f"Training Decision Tree for {disease_name}...")
            dt = DecisionTreeClassifier(random_state=42, max_depth=5)
            dt.fit(X_train, y_train)
            dt_pred = dt.predict(X_test)
            results['Decision Tree'] = {
                'model': dt,
                'predictions': dt_pred,
                'accuracy': accuracy_score(y_test, dt_pred),
                'precision': precision_score(y_test, dt_pred, average='weighted', zero_division=0),
                'recall': recall_score(y_test, dt_pred, average='weighted', zero_division=0),
                'f1': f1_score(y_test, dt_pred, average='weighted', zero_division=0),
                'confusion_matrix': confusion_matrix(y_test, dt_pred)
            }
        except Exception as e:
            print(f"❌ Error training Decision Tree: {e}")

        # 5. Random Forest
        try:
            print(f"Training Random Forest for {disease_name}...")
            rf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=5)
            rf.fit(X_train, y_train)
            rf_pred = rf.predict(X_test)
            results['Random Forest'] = {
                'model': rf,
                'predictions': rf_pred,
                'accuracy': accuracy_score(y_test, rf_pred),
                'precision': precision_score(y_test, rf_pred, average='weighted', zero_division=0),
                'recall': recall_score(y_test, rf_pred, average='weighted', zero_division=0),
                'f1': f1_score(y_test, rf_pred, average='weighted', zero_division=0),
                'confusion_matrix': confusion_matrix(y_test, rf_pred)
            }
        except Exception as e:
            print(f"❌ Error training Random Forest: {e}")

        # 6. Support Vector Machine
        try:
            print(f"Training SVM for {disease_name}...")
            svm = SVC(kernel='linear', probability=True, random_state=42)
            svm.fit(X_train, y_train)
            svm_pred = svm.predict(X_test)
            results['Support Vector Machine'] = {
                'model': svm,
                'predictions': svm_pred,
                'accuracy': accuracy_score(y_test, svm_pred),
                'precision': precision_score(y_test, svm_pred, average='weighted', zero_division=0),
                'recall': recall_score(y_test, svm_pred, average='weighted', zero_division=0),
                'f1': f1_score(y_test, svm_pred, average='weighted', zero_division=0),
                'confusion_matrix': confusion_matrix(y_test, svm_pred)
            }
        except Exception as e:
            print(f"❌ Error training SVM: {e}")

        # 7. Neural Network (MLP)
        try:
            print(f"Training Neural Network for {disease_name}...")
            nn = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)
            nn.fit(X_train, y_train)
            nn_pred = nn.predict(X_test)
            results['Neural Network'] = {
                'model': nn,
                'predictions': nn_pred,
                'accuracy': accuracy_score(y_test, nn_pred),
                'precision': precision_score(y_test, nn_pred, average='weighted', zero_division=0),
                'recall': recall_score(y_test, nn_pred, average='weighted', zero_division=0),
                'f1': f1_score(y_test, nn_pred, average='weighted', zero_division=0),
                'confusion_matrix': confusion_matrix(y_test, nn_pred)
            }
        except Exception as e:
            print(f"❌ Error training Neural Network: {e}")

        # 8. PCA for Dimensionality Reduction
        try:
            print(f"Applying PCA for {disease_name}...")
            pca = PCA(n_components=min(3, X_train.shape[1]))
            X_train_pca = pca.fit_transform(X_train)
            X_test_pca = pca.transform(X_test)

            # Train a model on PCA-reduced features
            lr_pca = LogisticRegression(max_iter=1000, random_state=42)
            lr_pca.fit(X_train_pca, y_train)
            lr_pca_pred = lr_pca.predict(X_test_pca)
            results['Logistic Regression (PCA)'] = {
                'model': lr_pca,
                'pca_model': pca,
                'predictions': lr_pca_pred,
                'accuracy': accuracy_score(y_test, lr_pca_pred),
                'precision': precision_score(y_test, lr_pca_pred, average='weighted', zero_division=0),
                'recall': recall_score(y_test, lr_pca_pred, average='weighted', zero_division=0),
                'f1': f1_score(y_test, lr_pca_pred, average='weighted', zero_division=0),
                'confusion_matrix': confusion_matrix(y_test, lr_pca_pred),
                'explained_variance': pca.explained_variance_ratio_.sum()
            }
            self.pca_models[disease_name] = pca
        except Exception as e:
            print(f"❌ Error training PCA model: {e}")

        # 9. K-Means Clustering (Unsupervised - for patient segmentation)
        try:
            print(f"Applying K-Means Clustering for {disease_name}...")
            kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
            if X_train.shape[1] > 10:
                pca_for_kmeans = PCA(n_components=2)
                X_reduced = pca_for_kmeans.fit_transform(X_train)
                kmeans.fit(X_reduced)
            else:
                kmeans.fit(X_train)
            self.kmeans_models[disease_name] = kmeans
        except Exception as e:
            print(f"❌ Error applying K-Means: {e}")

        return results

    def predict_single(self, model_name, disease_name, input_features):
        """Predict for a single patient"""
        if disease_name not in self.results:
            return "Model not trained for this disease"

        if model_name not in self.results[disease_name]:
            return "Model not found"

        model_info = self.results[disease_name][model_name]
        model = model_info['model']

        # Handle PCA models
        if 'pca_model' in model_info:
            pca = model_info['pca_model']
            input_pca = pca.transform([input_features])
            prediction = model.predict(input_pca)[0]
            probabilities = model.predict_proba(input_pca)[0]
        else:
            prediction = model.predict([input_features])[0]
            probabilities = model.predict_proba([input_features])[0]

        return {
            'prediction': int(prediction),
            'probabilities': probabilities.tolist(),
            'confidence': max(probabilities) * 100
        }

# =============================================
# 4. DATA VISUALIZATION FUNCTIONS (UNCHANGED)
# =============================================

def create_confusion_matrix_plot(cm, model_name):
    """Create a confusion matrix plot"""
    fig = go.Figure(data=go.Heatmap(
        z=cm,
        x=['Predicted Negative', 'Predicted Positive'],
        y=['Actual Negative', 'Actual Positive'],
        colorscale='Blues',
        text=cm,
        texttemplate='%{text}',
        textfont={"size": 16},
        hoverinfo='z'
    ))

    fig.update_layout(
        title=f'Confusion Matrix - {model_name}',
        xaxis_title='Predicted Label',
        yaxis_title='True Label',
        width=400,
        height=400
    )

    return fig

def create_metrics_bar_chart(metrics_dict, disease_name):
    """Create a bar chart comparing model metrics"""
    if not metrics_dict:
        # Create empty figure if no metrics
        fig = go.Figure()
        fig.update_layout(
            title=f'No models trained for {disease_name}',
            xaxis_title='Models',
            yaxis_title='Score',
            height=500
        )
        return fig

    models = list(metrics_dict.keys())
    accuracy = [metrics_dict[m]['accuracy'] for m in models]
    precision = [metrics_dict[m]['precision'] for m in models]
    recall = [metrics_dict[m]['recall'] for m in models]
    f1 = [metrics_dict[m]['f1'] for m in models]

    fig = go.Figure(data=[
        go.Bar(name='Accuracy', x=models, y=accuracy, marker_color='#1f77b4'),
        go.Bar(name='Precision', x=models, y=precision, marker_color='#ff7f0e'),
        go.Bar(name='Recall', x=models, y=recall, marker_color='#2ca02c'),
        go.Bar(name='F1-Score', x=models, y=f1, marker_color='#d62728')
    ])

    fig.update_layout(
        title=f'Model Performance Comparison - {disease_name}',
        xaxis_title='Models',
        yaxis_title='Score',
        barmode='group',
        height=500,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )

    return fig

def create_pca_variance_plot(pca_model, disease_name):
    """Create PCA explained variance plot"""
    if pca_model is None:
        # Create empty figure if no PCA model
        fig = go.Figure()
        fig.update_layout(
            title=f'No PCA model for {disease_name}',
            xaxis_title='Principal Components',
            yaxis_title='Explained Variance Ratio',
            height=400
        )
        return fig

    explained_variance = pca_model.explained_variance_ratio_
    cumulative_variance = np.cumsum(explained_variance)

    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=[f'PC{i+1}' for i in range(len(explained_variance))],
        y=explained_variance,
        name='Individual Variance',
        marker_color='lightblue'
    ))

    fig.add_trace(go.Scatter(
        x=[f'PC{i+1}' for i in range(len(cumulative_variance))],
        y=cumulative_variance,
        name='Cumulative Variance',
        marker_color='red',
        mode='lines+markers'
    ))

    fig.update_layout(
        title=f'PCA Explained Variance - {disease_name}',
        xaxis_title='Principal Components',
        yaxis_title='Explained Variance Ratio',
        height=400,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )

    return fig

# =============================================
# 5. MAIN APPLICATION CLASS
# =============================================

class HealthAnalysisApp:
    """Main application class integrating all components"""

    def __init__(self):
        self.datasets = {}
        self.predictor = DiseasePredictor()
        self.is_trained = False
        self.current_disease = None

        # Initialize all datasets
        self.init_datasets()

    def init_datasets(self):
        """Initialize all disease datasets"""
        print("Initializing disease datasets...")

        self.datasets['Heart Disease'] = HeartDiseaseDataset()
        self.datasets['Diabetes'] = DiabetesDataset()
        self.datasets['Stroke'] = StrokeDataset()
        self.datasets['Liver Disease'] = LiverDataset()
        self.datasets['Chronic Kidney Disease'] = KidneyDiseaseDataset()

        # Load and preprocess all datasets
        for name, dataset in self.datasets.items():
            print(f"\nProcessing {name}...")
            if dataset.load_data():
                if dataset.preprocess():
                    dataset.split_data()
                    dataset.scale_features()
                    print(f"✅ {name} dataset ready")
                else:
                    print(f"❌ Failed to preprocess {name} dataset")
            else:
                print(f"⚠️ Using dummy data for {name}")
                # Create and preprocess dummy data
                dataset.preprocess()
                dataset.split_data()
                dataset.scale_features()

    def train_all_models(self):
        """Train models for all diseases"""
        print("\n" + "="*50)
        print("TRAINING ALL MODELS")
        print("="*50)

        training_results = {}

        for disease_name, dataset in self.datasets.items():
            print(f"\nTraining models for {disease_name}...")

            if dataset.X_train is not None and dataset.y_train is not None:
                try:
                    results = self.predictor.train_models(
                        dataset.X_train, dataset.X_test,
                        dataset.y_train, dataset.y_test,
                        disease_name
                    )

                    self.predictor.results[disease_name] = results
                    if results:  # Check if any models were trained successfully
                        training_results[disease_name] = {
                            'status': 'Success',
                            'models_trained': len(results),
                            'best_accuracy': max([r['accuracy'] for r in results.values()])
                        }
                        print(f"✅ {disease_name}: {len(results)} models trained successfully")
                    else:
                        training_results[disease_name] = {
                            'status': 'Failed - No models could be trained',
                            'models_trained': 0,
                            'best_accuracy': 0
                        }
                        print(f"❌ {disease_name}: No models could be trained")
                except Exception as e:
                    print(f"❌ Error training models for {disease_name}: {e}")
                    training_results[disease_name] = {
                        'status': 'Failed',
                        'models_trained': 0,
                        'best_accuracy': 0,
                        'error': str(e)
                    }
            else:
                training_results[disease_name] = {
                    'status': 'Failed - No data',
                    'models_trained': 0,
                    'best_accuracy': 0
                }

        self.is_trained = True
        return training_results

    def get_disease_info(self, disease_name):
        """Get information about a specific disease"""
        if disease_name in self.datasets:
            dataset = self.datasets[disease_name]
            if dataset.data is not None:
                # Handle case where y might not be set yet
                if dataset.y is not None:
                    try:
                        class_dist = dict(dataset.y.value_counts())
                    except:
                        class_dist = {}
                else:
                    class_dist = {}
            else:
                class_dist = {}
            return {
                'name': disease_name,
                'samples': len(dataset.data) if dataset.data is not None else 0,
                'features': len(dataset.feature_names),
                'target_name': dataset.target_name,
                'class_distribution': class_dist,
                'status': 'Ready' if dataset.data is not None else 'Failed'
            }
        return None

    def predict_for_patient(self, disease_name, model_name, input_values):
        """Make prediction for a patient"""
        if not self.is_trained:
            return "Please train models first!"

        if disease_name not in self.predictor.results:
            return f"No models trained for {disease_name}"

        # Get the dataset for scaling
        dataset = self.datasets[disease_name]

        try:
            # Ensure input_values is a list
            if not isinstance(input_values, list):
                input_values = list(input_values)

            # Scale the input values
            input_scaled = dataset.scaler.transform([input_values])

            # Get prediction
            result = self.predictor.predict_single(model_name, disease_name, input_scaled[0])

            if isinstance(result, dict):
                # Format the result
                status = "POSITIVE" if result['prediction'] == 1 else "NEGATIVE"
                confidence = result['confidence']
                risk_level = "HIGH" if confidence > 70 else "MODERATE" if confidence > 50 else "LOW"

                return {
                    'status': status,
                    'risk_level': risk_level,
                    'confidence': f"{confidence:.2f}%",
                    'probabilities': result['probabilities'],
                    'recommendation': self.get_recommendation(disease_name, status, risk_level)
                }

            return result
        except Exception as e:
            return f"Prediction error: {str(e)}"

    def get_recommendation(self, disease_name, status, risk_level):
        """Get medical recommendation based on prediction"""
        recommendations = {
            'Heart Disease': {
                'POSITIVE_HIGH': "Immediate consultation with cardiologist required. Emergency services if chest pain.",
                'POSITIVE_MODERATE': "Schedule appointment with cardiologist within a week.",
                'POSITIVE_LOW': "Consult with primary care physician for further tests.",
                'NEGATIVE_HIGH': "Continue regular checkups and maintain heart-healthy lifestyle.",
                'NEGATIVE_MODERATE': "Maintain current lifestyle with regular monitoring.",
                'NEGATIVE_LOW': "Low risk. Continue preventive measures."
            },
            'Diabetes': {
                'POSITIVE_HIGH': "Immediate consultation with endocrinologist. Monitor blood sugar regularly.",
                'POSITIVE_MODERATE': "Schedule appointment with endocrinologist. Begin diet control.",
                'POSITIVE_LOW': "Consult with primary care physician for glucose tolerance test.",
                'NEGATIVE_HIGH': "Continue regular monitoring. Maintain healthy diet and exercise.",
                'NEGATIVE_MODERATE': "Regular checkups recommended. Watch for symptoms.",
                'NEGATIVE_LOW': "Low risk. Maintain healthy lifestyle."
            },
            'Stroke': {
                'POSITIVE_HIGH': "Immediate emergency care required if symptoms present.",
                'POSITIVE_MODERATE': "Schedule neurology consultation immediately.",
                'POSITIVE_LOW': "Consult with primary care physician for stroke risk assessment.",
                'NEGATIVE_HIGH': "Monitor blood pressure regularly. Maintain healthy lifestyle.",
                'NEGATIVE_MODERATE': "Regular checkups recommended.",
                'NEGATIVE_LOW': "Low risk. Continue preventive measures."
            },
            'Liver Disease': {
                'POSITIVE_HIGH': "Immediate hepatology consultation required. Avoid alcohol.",
                'POSITIVE_MODERATE': "Schedule appointment with hepatologist. Monitor liver enzymes.",
                'POSITIVE_LOW': "Consult with primary care physician for liver function tests.",
                'NEGATIVE_HIGH': "Maintain liver-healthy diet. Avoid hepatotoxic substances.",
                'NEGATIVE_MODERATE': "Regular checkups recommended.",
                'NEGATIVE_LOW': "Low risk. Maintain healthy lifestyle."
            },
            'Chronic Kidney Disease': {
                'POSITIVE_HIGH': "Immediate nephrology consultation required. Monitor kidney function.",
                'POSITIVE_MODERATE': "Schedule appointment with nephrologist. Monitor blood pressure.",
                'POSITIVE_LOW': "Consult with primary care physician for kidney function tests.",
                'NEGATIVE_HIGH': "Maintain kidney-healthy diet. Monitor regularly.",
                'NEGATIVE_MODERATE': "Regular checkups recommended.",
                'NEGATIVE_LOW': "Low risk. Stay hydrated and maintain healthy lifestyle."
            }
        }

        key = f"{status}_{risk_level}"
        if disease_name in recommendations and key in recommendations[disease_name]:
            return recommendations[disease_name][key]

        # Default recommendations
        if status == "POSITIVE":
            return f"Consult a healthcare professional for {disease_name} evaluation."
        else:
            return f"No immediate concern for {disease_name}. Continue preventive care."

    def get_model_metrics(self, disease_name):
        """Get metrics for all models of a disease"""
        if disease_name in self.predictor.results:
            metrics = {}
            for model_name, model_info in self.predictor.results[disease_name].items():
                metrics[model_name] = {
                    'accuracy': model_info['accuracy'],
                    'precision': model_info['precision'],
                    'recall': model_info['recall'],
                    'f1_score': model_info['f1']
                }
            return metrics
        return {}

# =============================================
# 6. GRADIO GUI INTERFACE (FIXED VERSION)
# =============================================

def create_gui_interface(app):
    """Create the Gradio GUI interface"""

    # Define CSS for styling
    css = """
    .gradio-container {
        max-width: 1200px !important;
        margin: auto !important;
    }
    .title {
        text-align: center;
        color: #2c3e50;
        font-size: 2.5em;
        margin-bottom: 20px;
    }
    .subtitle {
        text-align: center;
        color: #34495e;
        font-size: 1.5em;
        margin-bottom: 30px;
    }
    .section {
        background: #f8f9fa;
        padding: 20px;
        border-radius: 10px;
        margin-bottom: 20px;
        border: 1px solid #dee2e6;
    }
    .result-box {
        background: #e8f4fc;
        padding: 15px;
        border-radius: 8px;
        border-left: 5px solid #3498db;
        margin: 10px 0;
    }
    .warning {
        background: #fff3cd;
        border-left: 5px solid #ffc107;
    }
    .success {
        background: #d1ecf1;
        border-left: 5px solid #17a2b8;
    }
    """

    # Create the Gradio interface
    with gr.Blocks(css=css, theme=gr.themes.Soft()) as demo:

        # Header
        gr.Markdown("<div class='title'>🏥 Predictive Health Analysis System</div>")
        gr.Markdown("<div class='subtitle'>Chronic Disease Diagnosis using Machine Learning</div>")
        gr.Markdown("**Team:** Mustafa, Ali Naseer, Mohammad Osaibuddin Malik, Sufiyan")

        # Status indicator
        status_text = gr.Markdown("🚀 **Application Ready!** Click 'Train All Models' to start.")

        # Main tabs
        with gr.Tabs():

            # Tab 1: Model Training and Overview
            with gr.Tab("📊 Model Training & Overview"):
                with gr.Row():
                    with gr.Column(scale=1):
                        gr.Markdown("### Dataset Information")
                        disease_select = gr.Dropdown(
                            choices=list(app.datasets.keys()),
                            value="Heart Disease",
                            label="Select Disease"
                        )

                        # disease_info = gr.JSON(label="Dataset Details")

                        train_btn = gr.Button("🚀 Train All Models", variant="primary")
                        training_status = gr.JSON(label="Training Results")

                    with gr.Column(scale=2):
                        gr.Markdown("### Model Performance")
                        metrics_plot = gr.Plot(label="Model Comparison")
                        pca_plot = gr.Plot(label="PCA Analysis")

                # Update disease info when selection changes
            #    def update_disease_info(disease_name):
            #        info = app.get_disease_info(disease_name)
            #        return info

            #    disease_select.change(
            #        update_disease_info,
            #        inputs=[disease_select],
            #        outputs=[disease_info]
            #    )

                # Train models button - FIXED
                def train_models_and_update():
                    try:
                        results = app.train_all_models()

                        # Update status
                        status = "✅ Models trained successfully! You can now make predictions.\n\n"
                        for disease, result in results.items():
                            if result['status'] == 'Success':
                                status += f"✅ {disease}: {result['models_trained']} models trained (Best Acc: {result['best_accuracy']:.2%})\n"
                            else:
                                status += f"⚠️ {disease}: {result['status']}\n"

                        # Get current disease for plots
                        current_disease = disease_select.value

                        return results, status
                    except Exception as e:
                        error_msg = f"Error during training: {str(e)}"
                        print(error_msg)
                        return {"error": error_msg}, f"❌ {error_msg}"

                train_btn.click(
                    train_models_and_update,
                    outputs=[training_status, status_text]
                )

                # Update plots when disease changes - FIXED
                def update_plots(disease_name):
                    try:
                        if app.is_trained and disease_name in app.predictor.results:
                            metrics_fig = create_metrics_bar_chart(app.predictor.results.get(disease_name, {}), disease_name)
                            pca_fig = create_pca_variance_plot(app.predictor.pca_models.get(disease_name), disease_name)
                        else:
                            metrics_fig = create_metrics_bar_chart({}, disease_name)
                            pca_fig = create_pca_variance_plot(None, disease_name)
                        return metrics_fig, pca_fig
                    except Exception as e:
                        print(f"Error updating plots: {e}")
                        # Create empty plots
                        empty_fig = go.Figure()
                        empty_fig.update_layout(title=f"Error loading plots: {str(e)}", height=400)
                        return empty_fig, empty_fig

                disease_select.change(
                    update_plots,
                    inputs=[disease_select],
                    outputs=[metrics_plot, pca_plot]
                )

            # Tab 2: Patient Prediction - SIMPLIFIED AND FIXED
            with gr.Tab("👨‍⚕️ Patient Diagnosis"):
                with gr.Row():
                    with gr.Column(scale=1):
                        gr.Markdown("### Patient Information")
                        pred_disease_select = gr.Dropdown(
                            choices=list(app.datasets.keys()),
                            value="Heart Disease",
                            label="Select Disease for Diagnosis"
                        )

                        model_select = gr.Dropdown(
                            choices=[
                                "Logistic Regression",
                                "K-Nearest Neighbors",
                                "Naive Bayes",
                                "Decision Tree",
                                "Random Forest",
                                "Support Vector Machine",
                                "Neural Network",
                                "Logistic Regression (PCA)"
                            ],
                            value="Logistic Regression",
                            label="Select Model"
                        )

                        # Create input fields dynamically
                        input_components = {}
                        input_container = gr.Column()

                        # Function to create input fields for the selected disease
                        def create_inputs_for_disease(disease_name):
                            if disease_name in app.datasets:
                                dataset = app.datasets[disease_name]
                                sample_inputs = dataset.get_sample_inputs()

                                # Create new input components
                                inputs = []
                                for i, feature in enumerate(dataset.feature_names):
                                    default_val = sample_inputs.get(feature, 0)
                                    label = feature.replace('_', ' ').title()

                                    # Create appropriate input based on feature name
                                    if 'age' in feature.lower():
                                        inp = gr.Number(value=default_val, label=f"{label} (years)", minimum=0, maximum=120)
                                    elif 'sex' in feature.lower() or 'gender' in feature.lower():
                                        inp = gr.Number(value=default_val, label=f"{label} (0=Female, 1=Male)", minimum=0, maximum=1)
                                    elif 'bp' in feature.lower() or 'pressure' in feature.lower():
                                        inp = gr.Number(value=default_val, label=f"{label} (mmHg)", minimum=0, maximum=300)
                                    elif 'chol' in feature.lower() or 'glucose' in feature.lower():
                                        inp = gr.Number(value=default_val, label=f"{label} (mg/dL)", minimum=0, maximum=500)
                                    elif 'bmi' in feature.lower():
                                        inp = gr.Number(value=default_val, label=f"{label} (kg/m²)", minimum=10, maximum=60)
                                    else:
                                        inp = gr.Number(value=default_val, label=label)

                                    inputs.append(inp)
                                    input_components[feature] = inp

                                return inputs
                            return []

                        # Initial inputs
                        initial_inputs = create_inputs_for_disease("Heart Disease")

                        # Update inputs when disease changes
                        def update_inputs(disease_name):
                            new_inputs = create_inputs_for_disease(disease_name)
                            return new_inputs

                        # Connect disease selection to input update
                        pred_disease_select.change(
                            update_inputs,
                            inputs=[pred_disease_select],
                            outputs=[input_container]
                        )

                        predict_btn = gr.Button("🔍 Make Diagnosis", variant="primary")

                    with gr.Column(scale=1):
                        gr.Markdown("### Diagnosis Results")
                        diagnosis_result = gr.JSON(label="Prediction Results")

                        gr.Markdown("### Model Metrics")
                        model_metrics = gr.DataFrame(label="Performance Metrics")

                        gr.Markdown("### Confusion Matrix")
                        confusion_plot = gr.Plot(label="Confusion Matrix")

                # Prediction function - FIXED
                def make_prediction(disease_name, model_name, *input_values):
                    try:
                        if not app.is_trained:
                            return {
                                "error": "Please train models first using the Training tab!"
                            }, pd.DataFrame(), None

                        # Get the dataset for feature count
                        dataset = app.datasets[disease_name]
                        input_list = list(input_values)

                        # Ensure we have the right number of inputs
                        if len(input_list) > len(dataset.feature_names):
                            input_list = input_list[:len(dataset.feature_names)]

                        # Get prediction
                        result = app.predict_for_patient(disease_name, model_name, input_list)

                        if isinstance(result, str):
                            return {"error": result}, pd.DataFrame(), None

                        # Get model metrics
                        metrics = app.get_model_metrics(disease_name)
                        if model_name in metrics:
                            metrics_df = pd.DataFrame([metrics[model_name]])
                            metrics_df.index = [model_name]
                        else:
                            metrics_df = pd.DataFrame()

                        # Get confusion matrix
                        if (disease_name in app.predictor.results and
                            model_name in app.predictor.results[disease_name]):
                            cm = app.predictor.results[disease_name][model_name]['confusion_matrix']
                            cm_fig = create_confusion_matrix_plot(cm, model_name)
                        else:
                            cm_fig = None

                        return result, metrics_df, cm_fig
                    except Exception as e:
                        error_msg = f"Prediction failed: {str(e)}"
                        print(error_msg)
                        return {"error": error_msg}, pd.DataFrame(), None

                # Connect predict button - FIXED
                predict_btn.click(
                    make_prediction,
                    inputs=[pred_disease_select, model_select] + initial_inputs,
                    outputs=[diagnosis_result, model_metrics, confusion_plot]
                )

            # Tab 3: Algorithm Details (UNCHANGED)
            with gr.Tab("🤖 Algorithms Information"):
                gr.Markdown("""
                ## Machine Learning Algorithms Used

                ### 1. Logistic Regression
                - **Type**: Supervised Learning, Classification
                - **Purpose**: Predicts probability of binary outcome
                - **Use Case**: Disease presence/absence prediction

                ### 2. K-Nearest Neighbors (K-NN)
                - **Type**: Supervised Learning, Classification
                - **Purpose**: Classifies based on similarity to neighbors
                - **Use Case**: Patient similarity analysis

                ### 3. Naive Bayes
                - **Type**: Supervised Learning, Classification
                - **Purpose**: Probabilistic classifier based on Bayes theorem
                - **Use Case**: Text classification, medical diagnosis

                ### 4. K-Means Clustering
                - **Type**: Unsupervised Learning, Clustering
                - **Purpose**: Groups similar patients together
                - **Use Case**: Patient segmentation

                ### 5. Principal Component Analysis (PCA)
                - **Type**: Dimensionality Reduction
                - **Purpose**: Reduces feature space while preserving variance
                - **Use Case**: Feature extraction, visualization

                ### 6. Decision Tree
                - **Type**: Supervised Learning, Classification
                - **Purpose**: Tree-based classification with interpretable rules
                - **Use Case**: Medical decision rules

                ### 7. Random Forest
                - **Type**: Ensemble Learning, Classification
                - **Purpose**: Multiple decision trees for better accuracy
                - **Use Case**: Robust disease prediction

                ### 8. Support Vector Machine (SVM)
                - **Type**: Supervised Learning, Classification
                - **Purpose**: Finds optimal boundary between classes
                - **Use Case**: Complex classification tasks

                ### 9. Neural Network (MLP)
                - **Type**: Deep Learning, Classification
                - **Purpose**: Multi-layer perceptron for complex patterns
                - **Use Case**: Advanced pattern recognition
                """)

            # Tab 4: Team Information (UNCHANGED)
            with gr.Tab("👥 Team & Project Details"):
                gr.Markdown("""
                ## Project Team Members

                | Role | Member | ID |
                |------|--------|----|
                | Data Engineer | Mohammad Osaibuddin Malik | 02-131232-065 |
                | EDA & Visualization Specialist | Ali Naseer | 02-31232-080 |
                | Core ML Developer | Sufiyan | 02-31232-058 |
                | Advanced Algorithms Specialist | Mustafa | 02-31232-070 |

                ## Project Specifications
                - **University Level Project**: 30 marks each
                - **Total Lines of Code**: 1500+ lines
                - **Diseases Covered**: 5 chronic diseases
                - **Algorithms Implemented**: 9 machine learning models
                - **GUI Framework**: Gradio (Web-based interface)

                ## Technologies Used
                - Python 3.x
                - Scikit-learn for ML algorithms
                - Pandas & NumPy for data processing
                - Gradio for GUI
                - Plotly for visualizations

                ## Project Structure
                1. Data Loading & Preprocessing
                2. Exploratory Data Analysis
                3. Model Training & Evaluation
                4. GUI Interface for Prediction
                5. Results Visualization & Reporting
                """)

        # Initialize with first disease info
       # demo.load(
       #     lambda: app.get_disease_info("Heart Disease"),
       #     outputs=[disease_info]
       # )

        return demo

# =============================================
# 7. MAIN EXECUTION (UNCHANGED)
# =============================================

def main():
    """Main function to run the application"""
    print("="*70)
    print("PREDICTIVE HEALTH ANALYSIS SYSTEM - CHRONIC DISEASE DIAGNOSIS")
    print("="*70)
    print("\nInitializing application...")

    # Create application instance
    app = HealthAnalysisApp()

    # Create and launch GUI
    print("\nCreating GUI interface...")
    demo = create_gui_interface(app)

    # Launch the application
    print("\n" + "="*70)
    print("✅ Application ready!")
    print("⏳ Launching GUI...")
    print("="*70)

    # Launch the interface
    demo.launch(share=True, debug=False)

# =============================================
# 8. RUN THE APPLICATION (UNCHANGED)
# =============================================

if __name__ == "__main__":
    # Import additional libraries needed for Colab
    try:
        import google.colab
        IN_COLAB = True
        print("✅ Running in Google Colab environment")
        print("📦 Installing required packages...")
        import subprocess
        import sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gradio", "plotly", "pandas", "numpy", "scikit-learn", "matplotlib", "seaborn"])
        print("✅ Packages installed successfully!")
    except:
        IN_COLAB = False
        print("⚠️  Running in local environment")

    # Run the main function
    main()



✅ Running in Google Colab environment
📦 Installing required packages...
✅ Packages installed successfully!
PREDICTIVE HEALTH ANALYSIS SYSTEM - CHRONIC DISEASE DIAGNOSIS

Initializing application...
Initializing disease datasets...

Processing Heart Disease...
✅ Heart Disease dataset loaded successfully. Shape: (303, 14)
Using target column: target
✅ Heart Disease preprocessed. Features: 13
✅ Heart Disease data split: Train=(242, 13), Test=(61, 13)
✅ Heart Disease features scaled
✅ Heart Disease dataset ready

Processing Diabetes...
✅ Diabetes dataset loaded successfully. Shape: (768, 9)
Using target column: Outcome
✅ Diabetes preprocessed. Features: 8
✅ Diabetes data split: Train=(614, 8), Test=(154, 8)
✅ Diabetes features scaled
✅ Diabetes dataset ready

Processing Stroke...
✅ Stroke dataset loaded successfully. Shape: (5110, 12)
Using target column: stroke
✅ Stroke preprocessed. Features: 11
✅ Stroke data split: Train=(4088, 11), Test=(1022, 11)
✅ Stroke features scaled
✅ Stroke data